In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '16'
# 模块导入
from src.train_utils import (
    set_seed
)
# 设置随机种子
set_seed(42)

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"使用设备: {device}")
import gc

使用设备: cuda


In [2]:
## 加载模型配置

from pathlib import Path
print("加载配置...")
import yaml
model_config_path = '/root/lio/modelresearch/Configs/model_config.yaml'
train_config_path = '/root/lio/modelresearch/Configs/train_config.yaml'

with open(model_config_path, 'r', encoding='utf-8') as f:
    model_config = yaml.safe_load(f)
with open(train_config_path, 'r', encoding='utf-8') as f:
    train_config = yaml.safe_load(f)

model_version = model_config.get('model_version', 'multi_modal_model_1')
description = model_config.get('description')
print(f"model_version: {model_version}")
print(f"description: {description}")

## 保存配置
config_save_path = f"/root/lio/modelresearch/checkpoints/{model_version}/config"
config_save_path = Path(config_save_path)
# 确保保存目录存在
config_save_path.mkdir(exist_ok=True,parents=True)
## 保存model的config
model_config_save_path = f"{config_save_path}/model_config.yaml"
train_config_save_path = f"{config_save_path}/train_config.yaml"

with open(model_config_save_path, 'w') as f:
    yaml.dump(model_config, f, indent=4, sort_keys=False, allow_unicode=True)
with open(train_config_save_path, 'w') as f:
    yaml.dump(train_config, f, indent=4, sort_keys=False, allow_unicode=True)

加载配置...
model_version: TCN_multi_modal_model_4
description: 1.level_wise_encoder 编码,TCN 2.lob数据是2*40,使用revin 4.感受野128


### 创建数据集

In [3]:
from src.data.datamodule.data_loader import ETHUSDTDataLoaders

data_loaders = ETHUSDTDataLoaders()
 # 创建 DataLoader
print("创建 DataLoader...")
print(f"训练集大小: {len(data_loaders.train.dataset)}")
print(f"验证集大小: {len(data_loaders.val.dataset)}")

读取数据
lob (29369996, 43)
读取标签
(29369996, 6)
转换数据
lob (22463987, 5, 20)
转换标签
(22463987, 5)
读取数据
lob (29369996, 43)
读取标签
(29369996, 6)
转换数据
lob (6906009, 5, 20)
转换标签
(6906009, 5)
创建 DataLoader...
训练集大小: 224580
验证集大小: 69001


In [4]:
for inputs,labels in data_loaders.train:
    print(inputs['lob'].shape,labels.shape)
    break

torch.Size([128, 6000, 5, 20]) torch.Size([128, 5])


### 模型创建

##### 1.多模态模型

In [5]:
from src.models.TCN_lobencoder import LOB_TCN
# num_classes = data_loaders.train.dataset.num_classes
num_classes = 5
model = LOB_TCN(model_config,num_classes = num_classes)
model = model.to(device)

In [6]:

import torch
import torch.nn as nn
from torchinfo import summary
# 3. 构造输入张量（维度需匹配配置）
for inputs,labels in data_loaders.train:
    lob_input = inputs['lob']
    # trade_input = inputs['trade']
    break
# 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
class WrappedMultiModalModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
    
    def forward(self, lob, trade=None):
        # 构造模型需要的字典输入
        if trade is not None:
            inputs = {"lob": lob}
        else:
            inputs = {"lob": lob}
        return self.original_model(inputs)

# inputs = {'lob': lob_input}
# with torch.no_grad():
#     model(inputs)  # 这一步后，self.fusion 不再是 None
wrapped_model = WrappedMultiModalModel(model)

# 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）
summary(
    wrapped_model,
    input_data=[lob_input],  # 先lob，后trade
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    depth=5,  # 显示模型深度（层数）
    device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
)


Layer (type:depth-idx)                                                 Input Shape          Output Shape         Param #              Trainable
WrappedMultiModalModel                                                 [128, 6000, 5, 20]   [128, 5]             --                   True
├─LOB_TCN: 1-1                                                         [128, 6000, 5, 20]   [128, 5]             --                   True
│    └─RevIN2d: 2-1                                                    [128, 5, 6000, 20]   [128, 5, 6000, 20]   10                   True
│    └─LevelWiseEncoder: 2-2                                           [128, 5, 6000, 20]   [128, 6000, 16]      --                   True
│    │    └─LevelEncoder: 3-1                                          [128, 6000, 20, 5]   [128, 6000, 20, 16]  --                   True
│    │    │    └─Sequential: 4-1                                       [128, 6000, 20, 5]   [128, 6000, 20, 16]  --                   True
│    │    │    │    └─

In [7]:

# # 3. 构造输入张量（维度需匹配配置）
# batch_size = 2
# time_steps = all_config.get('data', {}).get('history_T', 3000)  # LOB/Trade的时间步必须一致
# lob_dim = model_config.get('lob_encoder', {}).get('in_channels', 4)
# # trade_dim = model_config.get('trade_encoder', {}).get('in_features', 12)
# trade_dim = 12
# lob_input = torch.randn(batch_size, lob_dim, time_steps, 10,device=device)  # (B,C,T,L) = (2,4,10,20)
# trade_input = torch.randn(batch_size, trade_dim, time_steps,device=device)    # (B,F,T) = (2,26,10)

# 4. 封装模型：将字典输入转为位置参数（适配torchinfo）
class WrappedMultiModalModel(nn.Module):
    def __init__(self, original_model):
        super().__init__()
        self.original_model = original_model
    
    def forward(self, lob):
        # 构造模型需要的字典输入
        inputs = {"lob": lob}
        return self.original_model(inputs)

# inputs = {'lob': lob_input}
# with torch.no_grad():
#     model(inputs)  # 这一步后，self.fusion 不再是 None
wrapped_model = WrappedMultiModalModel(model)
model_summary_str = str(summary(
    wrapped_model,
    input_data=[lob_input],
    col_names=["input_size", "output_size", "num_params", "trainable"],
    col_width=20,
    depth=4,
    device="cuda"
))
import matplotlib.pyplot as plt
# 2. 用 Matplotlib 绘制文本图
plt.figure(figsize=(20, 25))  # 根据模型长度调整
plt.text(0.01, 0.99, model_summary_str, fontsize=10, verticalalignment='top', family='monospace')
plt.axis('off')
plt.tight_layout()

# 3. 保存图片
model_summary_save_path = f"/root/lio/modelresearch/checkpoints/{model_version}"
model_summary_save_path = Path(model_summary_save_path)
plt.savefig(os.path.join(model_summary_save_path, f'{model_version}_model_summary.png'), 
            dpi=150, bbox_inches='tight')
plt.close()
# 5. 调用summary（核心：传入输入张量列表，顺序匹配封装模型的forward参数）,保存为图片
# summary_img = summary(
#     wrapped_model,
#     input_data=[lob_input, trade_input],  # 先lob，后trade
#     col_names=["input_size", "output_size", "num_params", "trainable"],
#     col_width=20,
#     depth=5,  # 显示模型深度（层数）
#     device="cuda"  # 若用GPU，改为"cuda"（需确保张量在GPU上）
# )
# summary_img.savefig(os.path.join(self.output_dir, f'{variant_name}_model_summary.png'))

In [8]:
# 打印模型信息
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"模型参数量: {total_params:,} (可训练: {trainable_params:,})")

模型参数量: 10,912 (可训练: 10,912)


### 训练

In [9]:
# import os
# import torch

# # # ===================== 日志关闭核心代码 (所有PyTorch版本通用) =====================
# # # 关闭 torch.compile 的 AUTOTUNE 满屏刷屏日志 (重中之重)
# # os.environ['TORCHINDUCTOR_PRINT_CONFIG'] = '0'
# # os.environ['TORCHINDUCTOR_VERBOSE'] = '0'
# # os.environ['TORCHINDUCTOR_AUTOTUNE_LOG'] = '0'
# # # 关闭PyTorch编译器的所有警告/错误日志输出
# # os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# # os.environ['TORCH_LOGS'] = '0'

# #'default'  , 'max-autotune'  , 'reduce-overhead'
# print("Compiling model...")
# model = torch.compile(model, mode='reduce-overhead') 

In [10]:
from src.trainer import Trainer
from src.train_utils import (
    setup_optimizer,
    setup_scheduler,
    setup_loss_functions,
    # set_seed
)

# 设置优化器和调度器
optimizer = setup_optimizer(model, train_config.get('optimizer', {}))
scheduler = setup_scheduler(optimizer, train_config.get('scheduler', {}))

# 设置损失函数
loss_fn = setup_loss_functions(train_config.get('loss', {}))


In [ ]:
# 创建训练器
Trainer_config = train_config.get('training', {})
trainer = Trainer(
    # model=model,
    # train_loader=train_loader,
    # val_loader=val_loader,
    loss_fn=loss_fn,
    optimizer=optimizer,
    scheduler=scheduler,
    config=Trainer_config,
    # device=device,
    variant_name=model_version, ## 用于保存模型
    seed=42
)

# 开始训练和验证
print("=" * 50)
history = trainer.fit(model=model,train_loader=data_loaders.train,val_loader=data_loaders.val)

## 保存为pickle
import pickle
history_save_path = f"/root/lio/modelresearch/checkpoints/{model_version}"
history_save_path = Path(history_save_path)
with open(os.path.join(history_save_path, f'{model_version}_history.pkl'), 'wb') as f:
    pickle.dump(history, f)

print('history保存成功')

print("=" * 50)
print("训练完成!")

# print(f"最佳验证 F1 (Up/Down): {max(history['val_f1_updown']):.4f}")

开始训练，共 10 个 epoch
设备: cuda
AMP: True
梯度累积步数: 1
--------------------------------------------------


Training:  60%|██████    | 1054/1754 [02:05<01:26,  8.06it/s, loss=0.00004756]

In [ ]:
history

{'train_history': [{'epoch': 1,
   'train_loss': 0.008953596185316394,
   'val_metrics': 0.00025735713291646944},
  {'epoch': 2,
   'train_loss': 8.102279886393022e-05,
   'val_metrics': 8.277610648683174e-05},
  {'epoch': 3,
   'train_loss': 1.963536270591963e-05,
   'val_metrics': 2.1214194694438032e-05},
  {'epoch': 4,
   'train_loss': 6.148153122015442e-06,
   'val_metrics': 3.507481456379438e-06},
  {'epoch': 5,
   'train_loss': 2.722954159752207e-06,
   'val_metrics': 1.8631783824118115e-06},
  {'epoch': 6,
   'train_loss': 1.961602522548164e-06,
   'val_metrics': 1.468799014791374e-06},
  {'epoch': 7,
   'train_loss': 1.718336943987895e-06,
   'val_metrics': 1.0688947926761258e-06},
  {'epoch': 8,
   'train_loss': 1.6210473665951161e-06,
   'val_metrics': 1.080119904851824e-06},
  {'epoch': 9,
   'train_loss': 1.6016948876669567e-06,
   'val_metrics': 1.0914468846520062e-06},
  {'epoch': 10,
   'train_loss': 1.611101845941936e-06,
   'val_metrics': 1.0247343540520642e-06}]}